# 5. Run the example model
**Before you start:** select the **Python (DECODE Intel)** kernel. Run each cell from top to bottom (Shift+Enter). If you change `config.yaml`, rerun the first cell to reload it. Nothing is saved unless a cell tells you it will save.
**Goal:** use the supplied network to locate molecules in DECODE’s example TIFF. You will see the input frame and predicted locations. This uses a small central crop so it is quicker on the Mac CPU. It does not train.

The example model and PSF were created for another microscope. Use this to learn the workflow; do not treat its predictions for your own data as validated.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import yaml

# Start Jupyter from the project folder or decode_walkthrough/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'decode_walkthrough' / 'config.yaml').is_file():
    ROOT = ROOT.parent
config_path = ROOT / 'decode_walkthrough' / 'config.yaml'
if not config_path.is_file():
    raise FileNotFoundError('Start Jupyter in the project folder or decode_walkthrough/.')
with config_path.open() as f:
    CFG = yaml.safe_load(f)
def project_path(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p
EXAMPLE = project_path(CFG['paths']['example_folder'])
WORK = project_path(CFG['paths']['work_folder'])
print('Project:', ROOT, '| Config:', config_path, '| Example exists:', EXAMPLE.exists())


In [ ]:
import torch, yaml, decode
import decode.utils.param_io
import decode.neuralfitter
import decode.neuralfitter.models
import decode.neuralfitter.utils.processing
import decode.neuralfitter.frame_processing
import decode.neuralfitter.scale_transform
import decode.neuralfitter.coord_transform
import decode.neuralfitter.post_processing
import decode.simulation.camera
import decode.utils.model_io

pred_cfg, own_cfg = CFG['prediction'], CFG['own_data']
USE_EXAMPLE = pred_cfg['use_example']
RAW_TIFF = project_path(own_cfg['raw_tiff'])
CHANNEL = own_cfg['channel']
if CHANNEL not in ('left', 'right'): raise ValueError('own_data.channel must be left or right')
channel_cfg = own_cfg['channels'][CHANNEL]
FIRST_FRAME, N_FRAMES = pred_cfg['first_frame'], pred_cfg['number_of_frames']
DEVICE = ('cuda:0' if torch.cuda.is_available() else 'cpu') if pred_cfg['device'] == 'auto' else pred_cfg['device']
if USE_EXAMPLE:
    MIRROR = pred_cfg['example_mirror']
    model_path = project_path(CFG['paths']['example_model'])
    param_path = project_path(CFG['paths']['example_parameters'])
else:
    MIRROR = channel_cfg['mirror']
    model_path = project_path(channel_cfg['model_file'])
    param_path = project_path(channel_cfg['model_parameters'])
    if any(x is None for x in (RAW_TIFF, model_path, param_path, MIRROR)):
        raise ValueError('Set raw_tiff, channel model_file, model_parameters and mirror in config.yaml.')



## A. Load a small piece of the example movie
The original example TIFF is single-channel. Your TIFF is split into left and right halves only when `prediction.use_example` is set to false.

In [ ]:
if FIRST_FRAME < 0 or N_FRAMES < 1: raise ValueError('Choose a valid positive frame range.')
path = project_path(CFG['paths']['example_movie']) if USE_EXAMPLE else RAW_TIFF
with tifffile.TiffFile(path) as tif:
    if FIRST_FRAME + N_FRAMES > len(tif.pages): raise ValueError('Requested frame range exceeds TIFF pages.')
    arr = np.stack([tif.pages[i].asarray() for i in range(FIRST_FRAME, FIRST_FRAME + N_FRAMES)])
if arr.ndim != 3: raise ValueError('Expected grayscale frames.')
x_offset = 0
if not USE_EXAMPLE:
    if CHANNEL not in ('left', 'right') or arr.shape[2] % 2: raise ValueError('Choose left/right and a TIFF with two equal halves.')
    width = arr.shape[2] // 2
    if CHANNEL == 'left': arr = arr[:, :, :width]
    else: arr, x_offset = arr[:, :, width:], width
if USE_EXAMPLE:
    # Small central region to make the CPU demonstration quick.
    h, w = arr.shape[-2:]
    crop_size = min(CFG['prediction']['example_crop_size'], h, w)
    top, left = (h-crop_size)//2, (w-crop_size)//2
    arr = arr[:, top:top+crop_size, left:left+crop_size]
    print('Example crop starts at (row, column):', (top, left))
print('Frames:', arr.shape, 'right-half raw x offset:', x_offset)
frames = torch.from_numpy(arr.astype(np.float32, copy=True))
plt.imshow(arr[0], cmap='gray'); plt.title('Selected channel, first frame'); plt.colorbar(label='ADU'); plt.show()


## B. Load the example network and its camera metadata
DECODE needs the model, the matching training parameters, and the camera conversion for the movie. The example movie’s EM gain differs from the training YAML, so it uses `frames_meta.yaml`.

In [ ]:
param = decode.utils.param_io.load_params(str(param_path))
model = decode.neuralfitter.models.SigmaMUNet.parse(param)
model = decode.utils.model_io.LoadSaveModel(model, input_file=model_path, output_file=None).load_init(device=DEVICE)
model = model.to(DEVICE)
if USE_EXAMPLE:
    with project_path(CFG['paths']['example_camera_metadata']).open() as f: camera_meta = yaml.safe_load(f)['Camera']
else:
    camera_meta = dict(channel_cfg['camera'])
    if any(camera_meta[k] is None for k in ('baseline', 'e_per_adu', 'spur_noise')):
        raise ValueError('Fill the selected channel camera settings in config.yaml before proceeding.')
param = decode.utils.param_io.RecursiveNamespace(**decode.utils.param_io.autofill_dict(camera_meta, param.to_dict(), mode_missing='include'))
camera = decode.simulation.camera.Photon2Camera.parse(param)
camera.device = 'cpu'


## C. Predict locations
The frames are converted to photon units, cropped to a multiple of 8, and mirrored according to the example convention. DECODE returns an emitter list, not a clean image. This may take a little time on the CPU.

In [ ]:
processing = decode.neuralfitter.utils.processing
steps = [processing.wrap_callable(camera.backward), decode.neuralfitter.frame_processing.AutoCenterCrop(8)]
if MIRROR: steps.append(decode.neuralfitter.frame_processing.Mirror2D(dims=-1))
steps.append(decode.neuralfitter.scale_transform.AmplitudeRescale.parse(param))
frame_proc = processing.TransformSequence(steps)
shape = decode.neuralfitter.frame_processing.get_frame_extent(frames.unsqueeze(1).size(), frame_proc.forward)
extent = ((-0.5, shape[-2]-0.5), (-0.5, shape[-1]-0.5))
print('Original H,W:', arr.shape[-2:], 'processed H,W:', shape[-2:])
print('Centre crop offsets (top,left):', ((arr.shape[-2]-shape[-2]+1)//2, (arr.shape[-1]-shape[-1]+1)//2), '| mirrored:', MIRROR)
post = processing.TransformSequence([
    decode.neuralfitter.scale_transform.InverseParamListRescale.parse(param),
    decode.neuralfitter.coord_transform.Offset2Coordinate(xextent=extent[0], yextent=extent[1], img_shape=shape[-2:]),
    decode.neuralfitter.post_processing.SpatialIntegration(raw_th=pred_cfg['threshold'], xy_unit='px', px_size=param.Camera.px_size)])
infer = decode.neuralfitter.Infer(model=model, ch_in=param.HyperParameter.channels_in, frame_proc=frame_proc, post_proc=post, device=DEVICE, batch_size=1, num_workers=pred_cfg['workers'])
emitter = infer.forward(frames)
print(emitter)


## D. Look at the result
Circles mark the model’s predicted locations; they are not ground truth. The two panels use DECODE’s processed frame (after crop and mirror), not the original TIFF orientation.

In [ ]:
# Plot the first processed frame and its estimated locations.
# Coordinates refer to the cropped/mirrored processed frame, not the original camera frame.
import decode.plot
subset = emitter.get_subset_frame(0, 0)
print('Localisations in first frame:', len(subset))
fig = plt.figure(figsize=(12,5))
plt.subplot(121); decode.plot.PlotFrameCoord(frame=frame_proc.forward(frames[[0]])).plot(); plt.title('Processed frame')
plt.subplot(122); decode.plot.PlotFrameCoord(frame=frame_proc.forward(frames[[0]]), pos_out=subset.xyz_px, phot_out=subset.prob).plot(); plt.title('Predicted localisations')
plt.show()


## E. Save only if needed
Set `prediction.save_output: true` in `config.yaml` and choose a fresh `prediction.output_file`. Coordinates refer to the processed crop; the notebook does not map them back to the original frame or register the channels.

**Your own data:** set `prediction.use_example: false`, fill the selected channel’s model, matching parameters, camera and mirror setting in `config.yaml`, and use a validated model. Images are split by channel and centre-cropped, but output coordinates still require careful mapping and validation. The example’s central 64-pixel crop is not applied to your frames.


In [ ]:
SAVE_OUTPUT = pred_cfg['save_output']
if SAVE_OUTPUT:
    WORK.mkdir(parents=True, exist_ok=True)
    output = project_path(pred_cfg['output_file'])
    if not output.parent.is_dir(): raise FileNotFoundError(f'Output directory does not exist: {output.parent}')
    if output.exists(): raise FileExistsError(output)
    emitter.save(str(output))
    print('Saved:', output, '; frame indices start at selected frame 0, original start:', FIRST_FRAME)
else:
    print('Set prediction.save_output: true in config.yaml after reviewing coordinates and filename.')
